# GNN training and review evidence
Run cells in order after setting `DATA_PREFIX` to the collector output prefix (without `_train.pkl`). Only load trusted pickle files. Original model architecture is retained.

Separate collector training and validation files are preserved. All generated frames are included, including successful E-hMP frames and SER=0. No oversampling or failure filtering is applied. Positive-class loss weight is calculated from training labels only.

Set optional TEST_PATH and TEST_METADATA_PATH only for an independently generated test split. Do not reuse validation as test. The supplied collector does not generate test files or retain transmitted-codeword IDs, so codeword-disjointness cannot be retrospectively certified. A future collector must retain codeword/noise hashes across all splits.

Required packages: `numpy pandas matplotlib torch scikit-learn galois`.
The collector metadata and both split files must share the same filename prefix.
Outputs are written to `review_evidence_cnn` or `review_evidence_gnn`.


In [ ]:
# Cell 1 - Imports
import os
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    average_precision_score,
)

print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
import json, hashlib, platform, time, sys
import sklearn
import galois


In [ ]:
# Paths and reproducible configuration
# DATA_PREFIX is the prefix printed by collect_training_data.py, without the
# _train.pkl / _validation.pkl / _metadata.pkl suffix. Point the notebook at a
# dataset through the environment instead of editing this cell:
#   AI_DECODER_DATA_DIR        directory holding the pickles   (default: data)
#   AI_DECODER_DATA_TIMESTAMP  timestamp in the file names
#   AI_DECODER_DATA_PREFIX     full prefix, overrides both of the above
DATA_DIR = Path(os.environ.get("AI_DECODER_DATA_DIR", "data"))
TIMESTAMP = os.environ.get("AI_DECODER_DATA_TIMESTAMP", "2026-09-05_11-52-08")
DATA_PREFIX = Path(os.environ.get(
    "AI_DECODER_DATA_PREFIX",
    DATA_DIR / f"error_location_BCH_31_21_r32_{TIMESTAMP}_residual",
))
TRAIN_PATH = Path(str(DATA_PREFIX) + "_train.pkl")
VALIDATION_PATH = Path(str(DATA_PREFIX) + "_validation.pkl")
METADATA_PATH = Path(str(DATA_PREFIX) + "_metadata.pkl")
TEST_PATH = None
TEST_METADATA_PATH = None
H_PATH = None  # Optional exact collector H saved as .npy; otherwise use the same BCH construction.
OUTPUT_DIR = Path("review_evidence_gnn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
TARGET_SEMANTICS = "post_ehmp_residual_error"
BATCH_SIZE = 32
NUM_WORKERS = 0  # avoids copying the large pickle dataset into Windows workers
EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
REPORTING_THRESHOLD = 0.5
HIDDEN_LAYER = 64
NUM_ITERS = 5
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
# Fixed epochs, no scheduler/early stopping; restore minimum validation-loss checkpoint.
# PyTorch default module initialization. Threshold fixed before validation/test evaluation.


## Collector data contract
`Y_bin`: post-E-hMP bits; `verified_list`: its verification mask;
`bin_total_check_node`: syndrome bits; `true_error_List`: **residual** symbol errors;
`SER`: generating channel SER. The companion metadata must declare
`target = "residual symbol error after E-hMP"`. Legacy injected-error data is rejected.

Every frame is checked for binary values, shapes, SER membership, and syndrome consistency with H.
The original pickle format loads all records into RAM; allow sufficient RAM for both large files.
Features are constructed per batch rather than caching a second full float feature dataset.


In [ ]:
# Cell 3 - Dataset helper functions and target contract

RESIDUAL_LABEL_FIELDS = (
    "residual_error_mask",
    "post_ehmp_error_mask",
    "target_residual_error",
)
LEGACY_INJECTED_LABEL_FIELDS = (
    "true_error_List",
    "injected_error_mask",
    "channel_error_mask",
    "e_true",
    "error_mask",
)


def pick(sample, names, required=True):
    for name in names:
        if name in sample:
            return sample[name]
    if required:
        raise KeyError(
            f"None of these fields were found: {names}. "
            f"Available: {list(sample.keys())}"
        )
    return None


def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _binary_vector(value, name):
    vector = to_numpy(value).astype(np.float32).reshape(-1)
    if not np.all(np.isin(vector, [0.0, 1.0])):
        raise ValueError(f"{name} must be a binary vector, got values {np.unique(vector)}")
    return vector


def _derive_residual_mask(sample):
    estimate = pick(
        sample,
        ["post_ehmp_bin", "decoded_bin", "decoder_output_bin", "V_hat_bin"],
        required=False,
    )
    transmitted = pick(
        sample,
        ["codeword_bin", "transmitted_bin", "V_bin"],
        required=False,
    )
    if estimate is None or transmitted is None:
        return None

    estimate = to_numpy(estimate)
    transmitted = to_numpy(transmitted)
    if estimate.shape != transmitted.shape:
        raise ValueError(
            "Post-E-hMP output and transmitted codeword must have the same shape; "
            f"got {estimate.shape} and {transmitted.shape}."
        )
    if estimate.ndim == 1:
        return (estimate != transmitted).astype(np.float32)
    if estimate.ndim == 2:
        return np.any(estimate != transmitted, axis=1).astype(np.float32)
    raise ValueError(
        "Stored post-E-hMP output/codeword must have shape [n] or [n,r], "
        f"got {estimate.shape}."
    )


def resolve_residual_label(sample, metadata):
    if "true_error_List" in sample:
        if metadata.get("target") != "residual symbol error after E-hMP":
            raise ValueError("true_error_List requires the new collector residual-target metadata")
        return _binary_vector(sample["true_error_List"], "true_error_List")
    for name in RESIDUAL_LABEL_FIELDS:
        if name in sample:
            return _binary_vector(sample[name], name)

    derived = _derive_residual_mask(sample)
    if derived is not None:
        return _binary_vector(derived, "derived residual_error_mask")

    for name in ("label", "labels"):
        if name in sample:
            semantics = sample.get(
                "label_semantics",
                metadata.get("label_semantics"),
            )
            if semantics != TARGET_SEMANTICS:
                raise ValueError(
                    f"Ambiguous field '{name}' is present, but label_semantics is "
                    f"{semantics!r}. Expected {TARGET_SEMANTICS!r}."
                )
            return _binary_vector(sample[name], name)

    legacy = [name for name in LEGACY_INJECTED_LABEL_FIELDS if name in sample]
    if legacy:
        raise ValueError(
            "Legacy injected-error label(s) detected: "
            f"{legacy}. These do not describe residual errors after E-hMP. "
            "Regenerate the dataset with 'residual_error_mask' computed by "
            "comparing the post-E-hMP symbol state with the transmitted codeword."
        )

    raise KeyError(
        "No residual target was found. Store 'residual_error_mask' or store both "
        "the post-E-hMP output and transmitted codeword so it can be derived."
    )


def build_cn_feature(check_info):
    """Return post-E-hMP check activity with shape [n-k, 1].

    If check_info is [n-k, r], each row becomes:
        1 = unsatisfied/nonzero check equation
        0 = satisfied/all-zero check equation
    """
    c = to_numpy(check_info)

    if c.ndim == 1:
        return (c != 0).astype(np.float32).reshape(-1, 1)
    if c.ndim == 2:
        if c.shape[1] == 1:
            return (c != 0).astype(np.float32)
        return np.any(c != 0, axis=1).astype(np.float32).reshape(-1, 1)
    raise ValueError(f"Unsupported check-node shape: {c.shape}")


def build_vn_feature(H, Y_bin, verified_mask):
    """VN feature = [post-E-hMP verification flag, r received bits, VN degree]."""
    H = to_numpy(H)
    Y_bin = to_numpy(Y_bin).astype(np.float32)
    verified_mask = _binary_vector(verified_mask, "verified_mask")

    if Y_bin.ndim != 2:
        raise ValueError(f"Y_bin must have shape [n,r], got {Y_bin.shape}")
    n, _ = Y_bin.shape
    if len(verified_mask) != n:
        raise ValueError("verified_mask length does not match n")
    if H.ndim != 2 or H.shape[1] != n:
        raise ValueError(f"H must have shape [n-k,n] with n={n}, got {H.shape}")

    degree = np.sum(H, axis=0).astype(np.float32).reshape(n, 1)
    verified = verified_mask.reshape(n, 1)
    return np.concatenate([verified, Y_bin, degree], axis=1).astype(np.float32)


In [ ]:
# Load independently stored splits without recombining them.
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

dataset_metadata = load_pickle(METADATA_PATH)
if dataset_metadata.get("target") != "residual symbol error after E-hMP":
    raise ValueError("Metadata does not declare the required residual target")
if dataset_metadata["train"]["seed"] == dataset_metadata["validation"]["seed"]:
    raise ValueError("Training and validation generation seeds must differ")
if TRAIN_PATH.resolve() == VALIDATION_PATH.resolve():
    raise ValueError("Training and validation must be separate files")
split_hashes = {"train": file_sha256(TRAIN_PATH), "validation": file_sha256(VALIDATION_PATH)}
if split_hashes["train"] == split_hashes["validation"]:
    raise ValueError("Train/validation files are byte-identical")
train_samples = load_pickle(TRAIN_PATH)
validation_samples = load_pickle(VALIDATION_PATH)
samples = train_samples
H_global = None
dataset_file = TRAIN_PATH
print("Loaded train/validation frames:", len(train_samples), len(validation_samples))


In [ ]:
# Cell 5 - Optional parity-check matrix H

# If H is not stored globally or inside each sample, load/set H here.
#
# Example:
# H_global = np.load("H_BCH_31_21.npy")
#
# The notebook needs H only when vn_feat is NOT already stored because
# variable-node degree is calculated as sum(H[:, i]).

# BCH construction comes from the shared decoder library so the notebook,
# the collector, and the evaluation script cannot drift apart.
from EhMP import BCH_sys

# Change BCH here only. The model dimensions are inferred automatically.
n = int(dataset_metadata["code"]["n"])
k = int(dataset_metadata["code"]["k"])

G, H_global = BCH_sys(n, k)
print("G shape:", G.shape, "H shape:", H_global.shape)

if H_global is not None:
    H_global = to_numpy(H_global)
    print("Global H shape:", H_global.shape)
else:
    print("No global H detected. H will be read from each sample if needed.")

if H_PATH is not None:
    H_global = np.load(H_PATH)
if H_global.shape != (n-k,n) or not np.isin(H_global,[0,1]).all():
    raise ValueError("Invalid binary parity-check matrix")


In [ ]:
# Cell 6 - Flexible GNN dataset with enforced residual targets

class GNNBCHDataset(Dataset):
    def __init__(self, samples, H_global=None, metadata=None):
        self.samples = samples
        self.H_global = H_global
        self.metadata = metadata or {}

        first_vn, first_cn, first_label = self._convert_sample(samples[0])
        self.n = first_vn.shape[0]
        self.vn_dim = first_vn.shape[1]
        self.cn_dim = first_cn.shape[1]
        self.r = self.vn_dim - 2

        if first_label.shape[0] != self.n:
            raise ValueError("Residual target length does not match n")

        print(
            f"Dataset detected: n={self.n}, r={self.r}, "
            f"vn_dim={self.vn_dim}, cn_dim={self.cn_dim}"
        )

    def _convert_sample(self, sample):
        if not isinstance(sample, dict):
            raise TypeError("Each sample must be a dictionary.")

        if "vn_feat" in sample:
            vn = to_numpy(sample["vn_feat"]).astype(np.float32)
        else:
            Y = pick(sample, ["Y_bin", "Y", "received_bin", "received"])
            verified = pick(
                sample,
                ["verified_mask", "verified_list", "phi", "varphi", "verified"],
            )
            H = sample.get("H", self.H_global)
            if H is None:
                raise KeyError(
                    "H is required to calculate VN degree. Store H in the "
                    "dataset/each sample or set H_global."
                )
            vn = build_vn_feature(H, Y, verified)

        if "cn_feat" in sample:
            cn = to_numpy(sample["cn_feat"]).astype(np.float32)
            if cn.ndim == 1:
                cn = cn.reshape(-1, 1)
        else:
            check_info = pick(
                sample,
                [
                    "bin_total_check_node", "S_check", "S", "check_node",
                    "check_nodes", "syndrome",
                ],
            )
            cn = build_cn_feature(check_info)

        label = resolve_residual_label(sample, self.metadata)
        if vn.ndim != 2 or cn.ndim != 2:
            raise ValueError("VN and CN features must both be two-dimensional.")
        if vn.shape[0] != label.shape[0]:
            raise ValueError(f"VN n={vn.shape[0]} but target n={label.shape[0]}")
        return vn, cn, label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vn, cn, label = self._convert_sample(self.samples[idx])
        return (
            torch.tensor(vn, dtype=torch.float32),
            torch.tensor(cn, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
        )


dataset = GNNBCHDataset(samples, H_global, dataset_metadata)

## GNN Input Features

The GNN-based error locator represents the BCH code as a Tanner graph constructed from the parity-check matrix. The graph consists of **variable nodes (VNs)** corresponding to the $n$ symbol positions and **check nodes (CNs)** corresponding to the $n-k$ parity-check equations.

The GNN receives three main inputs:

1. The parity-check matrix $\mathbf{H}$, which defines the graph connections.
2. The variable-node feature matrix $\mathbf{X}$.
3. The check-node feature matrix $\mathbf{C}$.

### 1. Parity-Check Matrix

The Tanner-graph connectivity is defined by the parity-check matrix

$$
\mathbf{H}
\in
\{0,1\}^{(n-k)\times n}.
$$

An element $H_{j,i}=1$ indicates that variable node $i$ is connected to check node $j$, while $H_{j,i}=0$ indicates that no connection exists.

For BCH$(31,21)$,

$$
n=31,
\qquad
n-k=10,
$$

and therefore

$$
\boxed{
\mathbf{H}\in\{0,1\}^{10\times31}
}.
$$

The matrix $\mathbf{H}$ is used during message passing to aggregate information between connected variable nodes and check nodes.

---

### 2. Variable-Node Features

For each symbol position $i$, the variable-node feature vector is defined as

$$
\mathbf{x}_i
=
\left[
\varphi_i,\,
y_{i,1},\,
y_{i,2},\,
\ldots,\,
y_{i,r},\,
w_i
\right],
$$

where:

- $\varphi_i$ is the verification flag of symbol $i$,
- $y_{i,1},y_{i,2},\ldots,y_{i,r}$ are the $r$ bits of the received symbol, and
- $w_i$ is the weight of the corresponding column of the parity-check matrix $\mathbf{H}$.

The variable-node weight is calculated as

$$
w_i
=
\sum_{j=1}^{n-k} H_{j,i}.
$$

Therefore, the dimension of each variable-node feature vector is

$$
d_{\mathrm{VN}}
=
1+r+1
=
r+2.
$$

For $r=32$,

$$
d_{\mathrm{VN}}=34.
$$

The features of all variable nodes are combined into the variable-node feature matrix

$$
\mathbf{X}
=
\begin{bmatrix}
\mathbf{x}_1\\
\mathbf{x}_2\\
\vdots\\
\mathbf{x}_n
\end{bmatrix}
\in
\mathbb{R}^{n\times(r+2)}.
$$

For BCH$(31,21)$ with $r=32$,

$$
\boxed{
\mathbf{X}
\in
\mathbb{R}^{31\times34}
}.
$$

For a batch containing $B$ samples, the variable-node input has the shape

$$
\boxed{
\mathbf{X}_{\mathrm{batch}}
\in
\mathbb{R}^{B\times31\times34}
}.
$$

---

### 3. Check-Node Features

Each check node contains information obtained from its corresponding parity-check equation. The check-node feature vector is represented as

$$
\mathbf{c}_j=[c_j],
$$

where $c_j$ indicates the state or activity of check node $j$ obtained from the verification-based decoding process.

The check-node features are combined into

$$
\mathbf{C}
=
\begin{bmatrix}
c_1\\
c_2\\
\vdots\\
c_{n-k}
\end{bmatrix}
\in
\mathbb{R}^{(n-k)\times1}.
$$

For BCH$(31,21)$,

$$
\boxed{
\mathbf{C}
\in
\mathbb{R}^{10\times1}
}.
$$

For a batch containing $B$ samples,

$$
\boxed{
\mathbf{C}_{\mathrm{batch}}
\in
\mathbb{R}^{B\times10\times1}
}.
$$

Unlike the CNN input, the check-node features are **not averaged into a single global feature**. They are maintained separately because each check node participates directly in the graph-based message-passing process.

---

### 4. GNN Input Summary

For BCH$(31,21)$ with $r=32$, the three GNN inputs are therefore

$$
\boxed{
\mathbf{H}\in\{0,1\}^{10\times31}
}
$$

$$
\boxed{
\mathbf{X}\in\mathbb{R}^{31\times34}
}
$$

and

$$
\boxed{
\mathbf{C}\in\mathbb{R}^{10\times1}.
}
$$

For a batch of $B$ samples, the model receives

$$
\mathbf{X}_{\mathrm{batch}}
\in
\mathbb{R}^{B\times31\times34},
$$

and

$$
\mathbf{C}_{\mathrm{batch}}
\in
\mathbb{R}^{B\times10\times1},
$$

while the same parity-check matrix $\mathbf{H}$ is used to define the Tanner-graph connectivity for all samples using the same BCH code.

These inputs allow the GNN to exchange information between variable nodes and check nodes according to the connections defined by $\mathbf{H}$ and to estimate the error probability of each symbol position.

In [ ]:
# Audit every stored frame, including successful baseline frames.
def audit_samples(name, records, metadata):
    if not isinstance(records, list) or not records:
        raise ValueError(f"{name}: expected a nonempty list of collector records")
    n, k, r = (int(metadata["code"][x]) for x in ("n", "k", "r"))
    grid = np.round(metadata["ser_values"], 6)
    groups = {}
    for sample in records:
        arrays = {key: np.asarray(sample[key]) for key in
                  ("Y_bin", "verified_list", "bin_total_check_node", "true_error_List")}
        for key, shape in {"Y_bin":(n,r), "verified_list":(n,),
                           "bin_total_check_node":(n-k,r), "true_error_List":(n,)}.items():
            a = arrays[key]
            if a.shape != shape or not np.isin(a, [0,1]).all():
                raise ValueError(f"{name}: invalid {key} shape or nonbinary values")
        if not np.array_equal((H_global.astype(np.int64) @ arrays["Y_bin"].astype(np.int64)) % 2,
                              arrays["bin_total_check_node"]):
            raise ValueError(f"{name}: syndrome disagrees with H and post-E-hMP Y")
        ser = round(float(sample["SER"]), 6)
        if ser not in grid:
            raise ValueError(f"{name}: unknown SER {ser}")
        row = groups.setdefault(ser, dict(split=name, SER=ser, frames=0, symbols=0,
             positive=0, negative=0, frames_with_residual_error=0,
             residual_at_verified_position=0, data_errors=0, data_unverified=0,
             data_error_frames=0))
        e = arrays["true_error_List"].astype(bool)
        verified = arrays["verified_list"].astype(bool)
        row["frames"] += 1; row["symbols"] += n
        row["positive"] += int(e.sum()); row["negative"] += int(n-e.sum())
        row["frames_with_residual_error"] += int(e.any())
        row["residual_at_verified_position"] += int((e & verified).sum())
        # Collector uses systematic H=[Q|I]: first k positions are data.
        row["data_errors"] += int(e[:k].sum())
        row["data_unverified"] += int((~verified[:k]).sum())
        row["data_error_frames"] += int(e[:k].any())
    table = pd.DataFrame(groups.values()).sort_values("SER")
    declared = metadata[name]
    if set(table.SER) != set(grid):
        raise ValueError(f"{name}: incomplete SER grid")
    for field, actual in [("num_frames",len(records)), ("positive_labels",int(table.positive.sum())),
                           ("negative_labels",int(table.negative.sum()))]:
        if int(declared[field]) != actual:
            raise ValueError(f"{name}: metadata count mismatch: {field}")
    if not (table.frames == int(declared["frames_per_ser"])).all():
        raise ValueError(f"{name}: per-SER frame counts differ from metadata")
    table["D_SER_baseline"] = table.data_errors / (table.frames*k)
    table["D_USR_baseline"] = table.data_unverified / (table.frames*k)
    table["P_data_error_baseline"] = table.data_error_frames / table.frames
    table.to_csv(OUTPUT_DIR / f"{name}_dataset_by_ser.csv",index=False)
    summary = {x:int(table[x].sum()) for x in table.select_dtypes("number").columns
               if x not in ("SER","D_SER_baseline","D_USR_baseline","P_data_error_baseline")}
    summary["prevalence"] = summary["positive"] / summary["symbols"]
    print(name, summary)
    return summary

if not np.array_equal(H_global[:, k:], np.eye(n-k)):
    raise ValueError("Baseline data metrics require collector H=[Q|I]")
train_audit = audit_samples("train", train_samples, dataset_metadata)
val_audit = audit_samples("validation", validation_samples, dataset_metadata)
train_dataset = dataset
val_dataset = type(dataset)(validation_samples, H_global, dataset_metadata)
train_size, val_size = len(train_dataset), len(val_dataset)
if train_audit["positive"] == 0 or train_audit["negative"] == 0:
    raise ValueError("Training requires both label classes")
POS_WEIGHT = train_audit["negative"] / train_audit["positive"]
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [ ]:
# GNN architecture matched to the original GNN notebook


if H_global is None:
    raise ValueError(
        "The GNN requires H_global because the Tanner graph is defined by H."
    )

H_MODEL = torch.tensor(
    H_global,
    dtype=torch.float32,
    device=DEVICE,
)

print("H shape:", tuple(H_MODEL.shape))


class FaultGNN(nn.Module):
    """
    Flexible GNN for BCH(n,k) and arbitrary r.

    VN feature dimension:
        vn_dim = r + 2

    Architecture:
        VN embedding
        -> repeated VN->CN aggregation
        -> CN update
        -> CN->VN aggregation
        -> residual VN update
        -> output classifier

    The original structure uses num_iters=5.
    """

    def __init__(
        self,
        vn_dim,
        cn_dim=1,
        hidden=64,
        num_iters=5
    ):
        super().__init__()

        self.num_iters = num_iters

        self.vn_embed = nn.Linear(vn_dim, hidden)

        self.cn_update = nn.Sequential(
            nn.Linear(hidden + cn_dim, hidden),
            nn.ReLU()
        )

        self.cn_to_vn = nn.Linear(hidden, hidden)

        self.vn_update = nn.Sequential(
            nn.Linear(hidden + hidden, hidden),
            nn.ReLU()
        )

        self.output = nn.Linear(hidden, 1)

    def forward(self, H, vn_feat, cn_feat):
        """
        H       : [m,n] or [B,m,n]
        vn_feat : [B,n,vn_dim]
        cn_feat : [B,m,cn_dim]

        output  : [B,n]
        """

        B = vn_feat.shape[0]

        if H.dim() == 2:
            H_batch = H.unsqueeze(0).expand(B, -1, -1)
        else:
            H_batch = H

        vn_hidden = self.vn_embed(vn_feat)  # [B,n,hidden]

        for _ in range(self.num_iters):

            # VN -> CN aggregation
            # [B,m,n] x [B,n,h] -> [B,m,h]
            cn_agg = torch.bmm(H_batch, vn_hidden)

            # Add check-node feature
            cn_input = torch.cat(
                [cn_agg, cn_feat],
                dim=-1
            )

            cn_hidden = self.cn_update(cn_input)

            # CN -> VN message
            cn_msg = self.cn_to_vn(cn_hidden)

            # [B,n,m] x [B,m,h] -> [B,n,h]
            vn_agg = torch.bmm(
                H_batch.transpose(1, 2),
                cn_msg
            )

            vn_input = torch.cat(
                [vn_agg, vn_hidden],
                dim=-1
            )

            # Residual VN update
            vn_hidden = (
                self.vn_update(vn_input)
                + vn_hidden
            )

        logits = self.output(vn_hidden).squeeze(-1)

        return logits


model = FaultGNN(
    vn_dim=dataset.vn_dim,
    cn_dim=dataset.cn_dim,
    hidden=HIDDEN_LAYER,
    num_iters=NUM_ITERS
).to(DEVICE)

print(model)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {num_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## GNN-Based Error Locator Model

The GNN-based error locator uses the Tanner-graph structure defined by the parity-check matrix $\mathbf{H}$ to exchange information between variable nodes (VNs) and check nodes (CNs). The model estimates the post-E-hMP residual-error probability of each symbol position by repeatedly performing message passing between the two node types.

For BCH$(31,21)$ with $r=32$, the model receives

$$
\mathbf{H}\in\{0,1\}^{10\times31},
$$

$$
\mathbf{X}\in\mathbb{R}^{31\times34},
$$

and

$$
\mathbf{C}\in\mathbb{R}^{10\times1},
$$

where $\mathbf{X}$ contains the variable-node features and $\mathbf{C}$ contains the check-node features.

The overall GNN architecture is

$$
\text{VN Embedding}
\rightarrow
\left[
\text{VN}\rightarrow\text{CN}
\rightarrow
\text{CN Update}
\rightarrow
\text{CN}\rightarrow\text{VN}
\rightarrow
\text{Residual VN Update}
\right]\times L
\rightarrow
\text{Classifier},
$$

where $L$ is the number of message-passing iterations. In this model,

$$
L=5.
$$

### 1. Variable-Node Embedding

The input variable-node feature matrix is first projected into a higher-dimensional hidden space using a fully connected layer,

$$
\mathbf{U}^{(0)}
=
\mathbf{X}\mathbf{W}_{e}
+
\mathbf{b}_{e},
$$

where $\mathbf{W}_{e}$ and $\mathbf{b}_{e}$ are trainable parameters.

The hidden feature dimension is

$$
D_h=64.
$$

Therefore, for BCH$(31,21)$,

$$
\boxed{
\mathbf{U}^{(0)}
\in
\mathbb{R}^{31\times64}
}.
$$

Each row $\mathbf{u}^{(0)}_i$ represents the initial hidden representation of variable node $i$.

---

### 2. Variable-Node to Check-Node Aggregation

During each message-passing iteration $l$, the hidden representations of the variable nodes are aggregated at the connected check nodes using the parity-check matrix,

$$
\mathbf{M}_{c}^{(l)}
=
\mathbf{H}\mathbf{U}^{(l)}.
$$

For BCH$(31,21)$,

$$
(10\times31)(31\times64)
\rightarrow
10\times64,
$$

and therefore

$$
\boxed{
\mathbf{M}_{c}^{(l)}
\in
\mathbb{R}^{10\times64}
}.
$$

Because $H_{j,i}=1$ only when variable node $i$ is connected to check node $j$, this operation aggregates information only from the variable nodes connected to each check node.

---

### 3. Check-Node Update

The aggregated variable-node information is concatenated with the corresponding check-node feature,

$$
\mathbf{A}_{c}^{(l)}
=
\left[
\mathbf{M}_{c}^{(l)},
\mathbf{C}
\right].
$$

Since the aggregated VN message has 64 features and each check node has one input feature, the input dimension of the check-node update is

$$
64+1=65.
$$

The check-node hidden representation is then calculated as

$$
\mathbf{C}^{(l)}
=
\mathrm{ReLU}
\left(
\mathbf{A}_{c}^{(l)}
\mathbf{W}_{c}
+
\mathbf{b}_{c}
\right),
$$

where $\mathbf{W}_{c}$ and $\mathbf{b}_{c}$ are trainable parameters.

The resulting check-node representation has the shape

$$
\boxed{
\mathbf{C}^{(l)}
\in
\mathbb{R}^{10\times64}
}.
$$

---

### 4. Check-Node to Variable-Node Message Passing

The updated check-node representations are first transformed using a trainable linear layer,

$$
\widetilde{\mathbf{C}}^{(l)}
=
\mathbf{C}^{(l)}\mathbf{W}_{cv}
+
\mathbf{b}_{cv}.
$$

The check-node information is then propagated back to the variable nodes using the transpose of the parity-check matrix,

$$
\mathbf{M}_{v}^{(l)}
=
\mathbf{H}^{\mathsf T}
\widetilde{\mathbf{C}}^{(l)}.
$$

For BCH$(31,21)$,

$$
(31\times10)(10\times64)
\rightarrow
31\times64,
$$

giving

$$
\boxed{
\mathbf{M}_{v}^{(l)}
\in
\mathbb{R}^{31\times64}
}.
$$

This operation allows each variable node to receive information from the check nodes to which it is connected.

---

### 5. Variable-Node Update with Residual Connection

The aggregated check-node message is concatenated with the current variable-node hidden representation,

$$
\widetilde{\mathbf{U}}^{(l)}
=
\left[
\mathbf{M}_{v}^{(l)},
\mathbf{U}^{(l)}
\right].
$$

Since both components contain 64 hidden features, the concatenated feature dimension is

$$
64+64=128.
$$

The variable-node update is calculated using

$$
\mathbf{F}^{(l)}
=
\mathrm{ReLU}
\left(
\widetilde{\mathbf{U}}^{(l)}
\mathbf{W}_{v}
+
\mathbf{b}_{v}
\right).
$$

A residual connection then adds the previous variable-node representation,

$$
\boxed{
\mathbf{U}^{(l+1)}
=
\mathbf{F}^{(l)}
+
\mathbf{U}^{(l)}
}.
$$

The residual connection preserves the previous variable-node information while allowing the GNN to learn additional information from the check-node messages.

The hidden representation remains

$$
\mathbf{U}^{(l+1)}
\in
\mathbb{R}^{31\times64}.
$$

This VN-to-CN and CN-to-VN message-passing process is repeated for $L=5$ iterations. The same trainable layers are reused at every iteration; the aggregations are unnormalized sums defined by $\mathbf H$ and $\mathbf H^{\mathsf T}$.

---

### 6. Error-Location Classifier

After the final message-passing iteration, the hidden representation of each variable node is passed through a linear output layer,

$$
o_i
=
\mathbf{u}^{(L)}_i
\mathbf{w}_{o}
+
b_o,
$$

where $o_i$ is the output logit for symbol $i$.

The complete output is

$$
\mathbf{o}
=
[o_1,o_2,\ldots,o_n].
$$

For BCH$(31,21)$,

$$
\boxed{
\mathbf{o}\in\mathbb{R}^{31}
}.
$$

The sigmoid function converts each logit into an estimated post-E-hMP residual-error probability,

$$
\hat{p}_i
=
\sigma(o_i)
=
\frac{1}{1+e^{-o_i}}.
$$

Therefore, the GNN produces

$$
\boxed{
\hat{\mathbf{p}}
=
[\hat{p}_1,\hat{p}_2,\ldots,\hat{p}_{31}]
},
$$

where $\hat{p}_i$ represents the estimated probability that symbol $i$ remains in error after E-hMP.

---

### 7. Error-Location Decision

A decision threshold $\tau$ can be applied to obtain the predicted error-location vector,

$$
\hat{e}_i =
\begin{cases}
1, & \hat{p}_i \geq \tau,\\
0, & \hat{p}_i < \tau.
\end{cases}
$$

The predicted probabilities can also be sorted in descending order to rank the unverified symbol positions according to their likelihood of remaining in error after E-hMP. These probabilities are subsequently used as guidance for the error-control decoding process.

In [ ]:
# Cell 9 - Test one forward pass

vn, cn, label = next(iter(train_loader))
vn = vn.to(DEVICE)
cn = cn.to(DEVICE)

H_MODEL = torch.tensor(
    H_global,
    dtype=torch.float32,
    device=DEVICE,
)

with torch.no_grad():
    logits = model(H_MODEL, vn, cn)

print("Input VN:", vn.shape)
print("Input CN:", cn.shape)
print("Output logits:", logits.shape)
print("Expected output: [batch, n]")


In [ ]:
# Cell 10 - Class-weighted residual-error objective and optimizer

# Calculate the imbalance weight from the training split only. This avoids
# leaking validation label statistics into model fitting.
pos_weight_tensor = torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("Target:", TARGET_SEMANTICS)
print("Loss:", criterion)
print("Positive-class weight:", float(pos_weight_tensor.cpu()))


## Model Evaluation

Evaluation compares the predicted probability with the **post-E-hMP residual
error mask**. A positive label means that the symbol remains wrong after the
baseline E-hMP pass.

The sigmoid output

$$
\hat p_i=\sigma(o_i)
$$

is the estimated residual-error probability. The default value
$\tau_{\mathrm{report}}=0.5$ converts probabilities to binary predictions only
for reporting accuracy, precision, recall, F1, and the confusion matrix. The
assisted decoder uses the continuous scores for ranking; this notebook does not
claim that 0.5 is an optimal decoding threshold.

Because residual errors are sparse, accuracy alone can be misleading. The
notebook therefore also reports threshold-independent ROC-AUC and PR-AUC
(average precision), together with positive-class precision and recall. All
metrics are computed over the validation symbols only; class weighting is
computed from the training split only.

In [ ]:
# Cell 11 - Residual-error evaluation function

@torch.no_grad()
def evaluate(model, loader, reporting_threshold=0.5):
    model.eval()
    losses = []
    all_labels = []
    all_probs = []

    for vn_feat, cn_feat, labels in loader:
        vn_feat = vn_feat.to(DEVICE)
        cn_feat = cn_feat.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(H_MODEL, vn_feat, cn_feat)
        loss = criterion(logits, labels)
        probs = torch.sigmoid(logits)

        losses.append((loss.item(), labels.numel()))
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    y_true_2d = np.concatenate(all_labels, axis=0)
    y_prob_2d = np.concatenate(all_probs, axis=0)
    y_pred_2d = (y_prob_2d >= reporting_threshold).astype(np.int32)

    y_true = y_true_2d.reshape(-1).astype(np.int32)
    y_prob = y_prob_2d.reshape(-1)
    y_pred = y_pred_2d.reshape(-1)

    metrics = {
        "loss": float(sum(loss * count for loss, count in losses) / sum(count for _, count in losses)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics["auc_roc"] = roc_auc_score(y_true, y_prob)
        metrics["auc_pr"] = average_precision_score(y_true, y_prob)
    except ValueError:
        metrics["auc_roc"] = np.nan
        metrics["auc_pr"] = np.nan

    return metrics, y_true_2d, y_prob_2d, y_pred_2d


In [ ]:
# Cell 12 - Training

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_auc_roc": [],
    "val_auc_pr": [],
}

best_val_loss = float("inf")
best_state = None
best_epoch = None
training_started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for vn_feat, cn_feat, labels in train_loader:
        vn_feat = vn_feat.to(DEVICE)
        cn_feat = cn_feat.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(H_MODEL, vn_feat, cn_feat)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.numel()

    train_loss = running_loss / (len(train_dataset) * dataset.n)

    val_metrics, _, _, _ = evaluate(
        model,
        val_loader,
        reporting_threshold=REPORTING_THRESHOLD,
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_auc_roc"].append(val_metrics["auc_roc"])
    history["val_auc_pr"].append(val_metrics["auc_pr"])

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.6f} | "
        f"val_loss={val_metrics['loss']:.6f} | "
        f"acc={val_metrics['accuracy']:.4f} | "
        f"precision={val_metrics['precision']:.4f} | "
        f"recall={val_metrics['recall']:.4f} | "
        f"F1={val_metrics['f1']:.4f} | "
        f"ROC-AUC={val_metrics['auc_roc']:.4f} | "
        f"PR-AUC={val_metrics['auc_pr']:.4f}"
    )

# Restore best validation model
if best_state is not None:
    model.load_state_dict(best_state)

print("Best validation loss:", best_val_loss)
training_seconds = time.perf_counter() - training_started
pd.DataFrame(history).rename_axis("epoch_zero_based").to_csv(OUTPUT_DIR / "training_history.csv")


In [ ]:
# Cell 13 - Plot training history

epochs_axis = np.arange(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(7, 5))
plt.plot(epochs_axis, history["train_loss"], marker="o", label="Training Loss")
plt.plot(epochs_axis, history["val_loss"], marker="s", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("GNN Residual-Error Locator - Training History")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(epochs_axis, history["val_precision"], label="Precision")
plt.plot(epochs_axis, history["val_recall"], label="Recall")
plt.plot(epochs_axis, history["val_f1"], label="F1")
plt.plot(epochs_axis, history["val_auc_roc"], label="ROC-AUC")
plt.plot(epochs_axis, history["val_auc_pr"], label="PR-AUC")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("GNN Residual-Error Locator - Validation Metrics")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 - Final validation evaluation

metrics, y_true, y_prob, y_pred = evaluate(
    model,
    val_loader,
    reporting_threshold=REPORTING_THRESHOLD,
)

print("Final validation results")
for key, value in metrics.items():
    print(f"{key:>10s}: {value:.6f}")

cm = confusion_matrix(y_true.reshape(-1), y_pred.reshape(-1), labels=[0,1])
print("\nConfusion matrix:")
print(cm)


## Inspect Residual-Error Scores for One Validation Frame

This sanity check shows the corrected target and the model scores for one
validation frame. `residual_error_target = 1` means that the corresponding
symbol remains incorrect after E-hMP. The table is sorted by the continuous
residual-error probability, which is the quantity used to rank candidate
positions for decoder assistance. `reported_residual_error` is included only
for classifier reporting at the selected reporting threshold.

In [ ]:
# Cell 16 - Inspect residual-error scores for one validation frame

model.eval()
vn_feat, cn_feat, labels = val_dataset[1]

with torch.no_grad():
    logits = model(
        H_MODEL,
        vn_feat.unsqueeze(0).to(DEVICE),
        cn_feat.unsqueeze(0).to(DEVICE),
    )
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()

result = pd.DataFrame({
    "symbol_index": np.arange(dataset.n),
    "residual_error_target": labels.numpy().astype(int),
    "residual_error_probability": probs,
})
result["reported_residual_error"] = (
    result["residual_error_probability"] >= REPORTING_THRESHOLD
).astype(int)
result = result.sort_values("residual_error_probability", ascending=False)
print(result.to_string(index=False))

In [ ]:
# Save PyTorch checkpoint with target and data provenance

k_detected = dataset.n - H_MODEL.shape[0]
MODEL_PATH = OUTPUT_DIR / f"gnn_error_locator_BCH_{dataset.n}_{int(k_detected)}_r{dataset.r}.pth"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_type": "gnn",
    "H": H_MODEL.detach().cpu().numpy(),
    "num_iters": NUM_ITERS,
    "n": dataset.n,
    "k": int(k_detected),
    "r": dataset.r,
    "vn_dim": dataset.vn_dim,
    "cn_dim": dataset.cn_dim,
    "hidden": HIDDEN_LAYER,
    "target_name": "residual_error_mask",
    "target_semantics": TARGET_SEMANTICS,
    "feature_state": "post_ehmp",
    "reporting_threshold": REPORTING_THRESHOLD,
    "threshold_role": "classification_reporting_only",
    "pos_weight": float(POS_WEIGHT),
    "dataset_filename": dataset_file.name,
    "dataset_metadata": dataset_metadata,
    "split_seed": SEED,
    "split_method": "collector separate files",
    "best_epoch": best_epoch,
    "validation_filename": VALIDATION_PATH.name,
    "train_audit": train_audit,
    "validation_audit": val_audit,
}

torch.save(checkpoint, MODEL_PATH)
print("Saved:", MODEL_PATH)


In [ ]:
# Reload PyTorch checkpoint

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,

)

if checkpoint.get("target_semantics") != TARGET_SEMANTICS:
    raise ValueError(
        "Checkpoint target semantics do not match the corrected residual-error "
        "definition. Retrain this model with the post-E-hMP dataset."
    )

loaded_model = FaultGNN(
    vn_dim=checkpoint["vn_dim"],
    cn_dim=checkpoint["cn_dim"],
    hidden=checkpoint["hidden"],
    num_iters=checkpoint["num_iters"],
).to(DEVICE)

loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

H_LOADED = torch.tensor(
    checkpoint["H"],
    dtype=torch.float32,
    device=DEVICE,
)

print("Reloaded:", MODEL_PATH)

In [ ]:
# Export classifier evidence (all n symbols pooled; never conditioned on failure).
def classifier_row(labels, probabilities):
    y = np.asarray(labels).reshape(-1).astype(np.uint8)
    p = np.asarray(probabilities).reshape(-1)
    pred = p >= REPORTING_THRESHOLD
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    both = np.unique(y).size == 2
    return dict(symbols=int(y.size), positive=int(y.sum()), negative=int(y.size-y.sum()),
        TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
        accuracy=float(accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)),
        recall=float(recall_score(y,pred,zero_division=0)),
        f1=float(f1_score(y,pred,zero_division=0)),
        auc_roc=float(roc_auc_score(y,p)) if both else None,
        average_precision=float(average_precision_score(y,p)) if both else None,
        brier_score=float(np.mean((p.astype(np.float64)-y)**2)))

def export_classifier(split, records, labels, probabilities, loss):
    ser = np.round([float(s["SER"]) for s in records],6)
    rows = [dict(split=split,SER="all",frames=len(records),weighted_bce=loss,
                 **classifier_row(labels,probabilities))]
    for value in np.unique(ser):
        keep = ser == value
        rows.append(dict(split=split,SER=float(value),frames=int(keep.sum()),
                          **classifier_row(labels[keep],probabilities[keep])))
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / f"{split}_classifier_metrics.csv",index=False)
    # Fixed equal-width reliability bins, before any calibration fitting.
    calibration = []
    for value in [None,*np.unique(ser)]:
        keep = np.ones(len(ser),dtype=bool) if value is None else ser == value
        y = labels[keep].reshape(-1); p = probabilities[keep].reshape(-1)
        bins = np.minimum((p*10).astype(int),9)
        for j in range(10):
            mask = bins == j
            calibration.append(dict(split=split,SER="all" if value is None else float(value),
                bin_lower=j/10,bin_upper=(j+1)/10,count=int(mask.sum()),
                mean_probability=float(p[mask].mean()) if mask.any() else None,
                observed_error_rate=float(y[mask].mean()) if mask.any() else None))
    pd.DataFrame(calibration).to_csv(OUTPUT_DIR / f"{split}_calibration.csv",index=False)
    return rows[0]

validation_report = export_classifier("validation", validation_samples, y_true, y_prob, metrics["loss"])
provenance = {name:dict(path=str(path.resolve()),sha256=split_hashes[name] if name in split_hashes else file_sha256(path))
              for name,path in [("train",TRAIN_PATH),("validation",VALIDATION_PATH),("metadata",METADATA_PATH)]}
if provenance["train"]["sha256"] == provenance["validation"]["sha256"]:
    raise ValueError("Train/validation files are byte-identical")

# Test is deliberately evaluated only after model selection and checkpoint saving.
test_report = None
if TEST_PATH is not None:
    if TEST_METADATA_PATH is None:
        raise ValueError("Independent test metadata is required")
    test_metadata = load_pickle(TEST_METADATA_PATH)
    if test_metadata.get("target") != dataset_metadata["target"] or test_metadata["code"] != dataset_metadata["code"]:
        raise ValueError("Test target/code do not match")
    test_seed = test_metadata["test"]["seed"]
    if test_seed in [dataset_metadata[x]["seed"] for x in ("train","validation")]:
        raise ValueError("Test generation seed must be independent")
    test_hash = file_sha256(TEST_PATH)
    if test_hash in [provenance[x]["sha256"] for x in ("train","validation")]:
        raise ValueError("Test file duplicates a fitting split")
    test_samples = load_pickle(TEST_PATH)
    audit_samples("test",test_samples,test_metadata)
    test_dataset = type(dataset)(test_samples,H_global,test_metadata)
    test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
    test_metrics,test_y,test_p,_ = evaluate(model,test_loader,REPORTING_THRESHOLD)
    test_report = export_classifier("test",test_samples,test_y,test_p,test_metrics["loss"])
    provenance["test"] = dict(path=str(Path(TEST_PATH).resolve()),sha256=test_hash,seed=test_seed)

configuration = dict(model_type=checkpoint["model_type"],seed=SEED,
    generation_metadata=dataset_metadata,files=provenance,
    batch_size=BATCH_SIZE,epochs_requested=EPOCHS,epochs_completed=len(history["train_loss"]),
    best_epoch=best_epoch,selection="min_validation_weighted_bce",threshold=REPORTING_THRESHOLD,
    threshold_role="classification_reporting_only",
    optimizer="Adam",learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,
    adam_betas=list(optimizer.defaults["betas"]),adam_eps=optimizer.defaults["eps"],
    scheduler=None,early_stopping=None,initialization="pytorch_default",
    loss="BCEWithLogitsLoss",loss_reduction=criterion.reduction,pos_weight=float(POS_WEIGHT),
    resampling=False,pos_weight_source="training_labels",
    failure_filtering=False,training_seconds=training_seconds,
    parameters=sum(p.numel() for p in model.parameters()),architecture=str(model),
    python=sys.version,numpy=np.__version__,pytorch=torch.__version__,sklearn=sklearn.__version__,galois=galois.__version__,
    platform=platform.platform(),processor=platform.processor(),device=str(DEVICE),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),
    deterministic_algorithms=torch.are_deterministic_algorithms_enabled(),
    deterministic_warn_only=torch.is_deterministic_algorithms_warn_only_enabled(),
    data_workers=NUM_WORKERS,validation=validation_report,test=test_report)
with open(OUTPUT_DIR / "run_configuration.json","w") as f:
    json.dump(configuration,f,indent=2)
np.save(OUTPUT_DIR / "H_used.npy",H_global)
print("Evidence saved to",OUTPUT_DIR.resolve())